In [ ]:
from setfit import SetFitModel, SetFitTrainer
from datasets import Dataset
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
df = pd.read_excel("data.xlsx") 
df['category'] = df['category'].str.strip() 

In [ ]:
counts = df['category'].value_counts()
print(counts)

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["category"],
    random_state=42
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(train_dataset)
print(test_dataset)

In [ ]:
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-mpnet-base-v2") 

In [ ]:
model.to("cuda")

In [ ]:
trainer = SetFitTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    batch_size=8,        
    num_iterations=10,    
    num_epochs=1,       
    column_mapping={"title": "text", "category": "label"},
)

In [ ]:
trainer.train()

In [ ]:
preds = trainer.model.predict(test_dataset["title"])
labels = test_dataset["category"]

acc = accuracy_score(labels, preds)
prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-score: {f1:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Confusion matrix
cm = confusion_matrix(labels, preds, labels=sorted(set(labels)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(set(labels)))

# Display
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True)
plt.title("Confusion matrix")
plt.show()


In [ ]:
trainer.model.save_pretrained("my_model_setfit")